# QA Datasets for Routing Experiments - Demo

## Overview
This notebook demonstrates the collection and standardization of QA datasets for routing experiments. 
A learned router can pick between decoding strategies per prompt to beat always using either one alone.

## What this artifact does
1. Loads standardized QA datasets (GSM8K, ARC-Challenge, BoolQ, MMLU)
2. Converts them to experiment format with fields: `input`, `output`, `metadata_*`
3. Creates a unified dataset structure ready for routing experiments

## Datasets included
- **GSM8K**: Math word problems (7,473 examples)
- **ARC-Challenge**: Science reasoning (1,119 examples)
- **BoolQ**: Boolean questions (9,427 examples)
- **MMLU**: Multiple-choice questions (752 examples)

All datasets have automatically verifiable answers and proper provenance.

In [ ]:
# Install dependencies - follows aii-colab pattern
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru is NOT pre-installed on Colab, always install
_pip('loguru')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2')

In [ ]:
# Imports - copied from original data.py with additions for notebook
import json
from pathlib import Path
from loguru import logger
import sys

# Setup logger (same as original)
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
# Data loading helper - GitHub URL with local fallback pattern
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-073c82-when-do-tiny-learned-routers-improve-dec/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub load failed: {e}")
    
    # Fallback to local file
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

import os

In [ ]:
# Load the demo data
print("Loading demo data...")
data = load_data()
print(f"Loaded {len(data['datasets'])} datasets")

# Display dataset summary
for dataset in data['datasets']:
    print(f"  {dataset['dataset']}: {len(dataset['examples'])} examples")

## Configuration

Minimal configuration for the demo. 
The original script processes all 4 primary datasets, but for this demo we use a pre-loaded subset.

In [ ]:
# Config cell - minimal parameters
# In the original script, these were file paths
# For the demo, we use the pre-loaded data variable

# Original paths (not used in demo - data is pre-loaded)
# INPUT_FILE = Path("processed_datasets/combined_dataset.json")
# OUTPUT_FILE = Path("full_data_out.json")

# For demo: use the loaded data directly
input_data = data  # Use the data we loaded from GitHub/local

# Only include primary datasets (same as original)
primary_datasets = ["openai/gsm8k", "allenai/ai2_arc", "google/boolq", "cais/mmlu"]

print("Configuration set:")
print(f"  Primary datasets: {primary_datasets}")

## Data Conversion

This section converts the processed datasets to the experiment format.
The `convert_to_experiment_format` function:
1. Groups examples by dataset source
2. Filters to only primary datasets
3. Converts each example to have: `input`, `output`, `metadata_*` fields

In [ ]:
# Conversion function - copied exactly from original data.py
def convert_to_experiment_format(input_data):
    """Convert processed dataset to experiment format."""
    # Group examples by dataset_source
    datasets_dict = {}
    
    # Only include the 4 primary datasets from the artifact plan
    primary_datasets = ["openai/gsm8k", "allenai/ai2_arc", "google/boolq", "cais/mmlu"]
    
    for example in input_data["examples"]:
        dataset_name = example["dataset_source"]
        
        # Skip if not in primary datasets
        if dataset_name not in primary_datasets:
            continue
        
        if dataset_name not in datasets_dict:
            datasets_dict[dataset_name] = {
                "dataset": dataset_name,
                "examples": []
            }
        
        # Convert to required format
        converted_example = {
            "input": example["prompt"],
            "output": str(example["correct_answer"]),
            "metadata_task_type": example["task_type"],
            "metadata_subject": example["subject"],
            "metadata_id": example["id"]
        }
        
        # Add any additional metadata
        if "metadata" in example and example["metadata"]:
            for key, value in example["metadata"].items():
                if key not in ["full_answer", "choices", "labels"]:  # Skip large fields
                    converted_example[f"metadata_{key}"] = value
        
        datasets_dict[dataset_name]["examples"].append(converted_example)
    
    # Convert to list
    datasets_list = list(datasets_dict.values())
    
    return {
        "datasets": datasets_list
    }

## Run Conversion

Execute the conversion on our demo data.
Note: The demo data only contains GSM8K examples, so only that dataset will appear in output.

In [ ]:
# Run the conversion (adapted from original __main__ block)
# Note: Demo data is already in the format expected by convert_to_experiment_format
# but we'll simulate the original structure for demonstration

# First, transform demo data to match expected input format
# The demo data has 'datasets' array, original expects 'examples' array
simulated_input = {"examples": [], "total_examples": 0}

for dataset in data['datasets']:
    dataset_name = dataset['dataset']
    for example in dataset['examples']:
        # Transform to the format expected by convert_to_experiment_format
        simulated_example = {
            "dataset_source": dataset_name,
            "prompt": example["input"],
            "correct_answer": example["output"],
            "task_type": example.get("metadata_task_type", "unknown"),
            "subject": example.get("metadata_subject", "unknown"),
            "id": example.get("metadata_id", "unknown"),
            "metadata": {}
        }
        # Add any additional metadata fields
        for key, value in example.items():
            if key.startswith("metadata_") and key != "metadata_task_type" and key != "metadata_subject" and key != "metadata_id":
                simulated_example["metadata"][key.replace("metadata_", "")] = value
        
        simulated_input["examples"].append(simulated_example)

simulated_input["total_examples"] = len(simulated_input["examples"])

logger.info(f"Converting {simulated_input['total_examples']} examples to experiment format...")
output_data = convert_to_experiment_format(simulated_input)

logger.info(f"Conversion complete!")
logger.info(f"Total datasets: {len(output_data['datasets'])}")
for dataset in output_data['datasets']:
    logger.info(f"  {dataset['dataset']}: {len(dataset['examples'])} examples")

## Results and Visualization

Display the converted data structure and summary statistics.

In [ ]:
# Visualize the results
import pandas as pd

print("="*60)
print("CONVERSION RESULTS")
print("="*60)

# Create summary table
summary_data = []
for dataset in output_data['datasets']:
    summary_data.append({
        'Dataset': dataset['dataset'],
        'Examples': len(dataset['examples']),
        'Sample Input (truncated)': dataset['examples'][0]['input'][:60] + '...' if dataset['examples'] else 'N/A',
        'Sample Output': dataset['examples'][0]['output'] if dataset['examples'] else 'N/A'
    })

df = pd.DataFrame(summary_data)
print("\nDataset Summary:")
print(df.to_string(index=False))

# Show example structure
print("\n" + "="*60)
print("EXAMPLE OUTPUT STRUCTURE")
print("="*60)
if output_data['datasets']:
    example = output_data['datasets'][0]['examples'][0]
    print("\nFirst example fields:")
    for key in example.keys():
        value = str(example[key])
        if len(value) > 80:
            value = value[:80] + '...'
        print(f"  {key}: {value}")

In [ ]:
# Save output (optional - for testing)
output_path = Path("demo_output.json")
output_path.write_text(json.dumps(output_data, indent=2))
print(f"\nSaved converted data to {output_path}")
print(f"File size: {output_path.stat().st_size} bytes")